In [1]:
from google.colab import files
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
import csv

In [3]:
uploaded_files = files.upload()

data=np.genfromtxt('Dedup_4vert2Max5MutationsAcyclicLabeled.csv',delimiter=',')

Saving Dedup_4vert2Max5MutationsAcyclicLabeled.csv to Dedup_4vert2Max5MutationsAcyclicLabeled.csv


In [4]:
SIZE = 295502
data = np.zeros((295502, 16 + 8))

def ternary_to_int(strT):
  if strT=="T": return 1.0
  elif strT=="F": return 0.0
  else: return 0.5

row_parse = lambda r : ([int(x) for x in r["quiver exchange matrix"][1:-1].split(" ")] +
                        [ternary_to_int(r["mutation finite"])] + [ternary_to_int(r["surface quiver"])] +
                        [int(r["determinant of exchange matrix"])] +  [int(r["determinant of companion modulo 4"])] +
                        [int(r["rank of exchange matrix"])] +  [ternary_to_int(r["mutation equivalent to a quiver with a mutation-cyclic 3-vertex subquiver"])] +
                        [int(r["minimum number of arrows in class"])] + [ternary_to_int(r["mutation acyclic"])])

In [5]:
with open('Dedup_4vert2Max5MutationsAcyclicLabeled.csv', 'r', newline='') as f:
  reader = csv.DictReader(f)
  header = reader.fieldnames

  for i, row in enumerate(reader):
    data[i,:] = row_parse(row)

In [6]:
data.shape

(295502, 24)

In [7]:
t = data[:,-1]
Quivers = data[:,:16]

In [8]:
X_train, X_test, t_train, t_test = train_test_split(Quivers, t, test_size=0.3)

In [9]:
num_classes = 2
input_shape = (16,)

X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)
print("X_train shape:", X_train.shape)
print(X_train.shape[0], "train samples")
print(X_test.shape[0], "test samples")

X_train shape: (206851, 16, 1)
206851 train samples
88651 test samples


In [10]:

# convert class vectors to binary class matrices
t_train = keras.utils.to_categorical(t_train, num_classes)
t_test = keras.utils.to_categorical(t_test, num_classes)

In [11]:
max_faces = 128
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Dense(max_faces, activation='sigmoid'),
        layers.Dense(max_faces, activation='sigmoid'),
        layers.Dense(max_faces, activation='sigmoid'),
        layers.Dense(num_classes, activation="sigmoid", name="all_active"),
    ]
)

In [12]:
model.get_layer(name="all_active").trainable=False
model.get_layer('all_active').set_weights([np.array([[1.0]*max_faces, [-1.0]*max_faces]).T, np.array([-max_faces +0.5, max_faces - 0.5])])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ all_active (Dense)              │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,458 (138.51 KB)

 Trainable params: 35,200 (137.50 KB)

 Non-trainable params: 258 (1.01 KB)

In [13]:
batch_size = 100
epochs = 1000

optimizer = keras.optimizers.Adam(learning_rate=0.005)

model.compile(loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

model.fit(X_train, t_train, batch_size=batch_size, epochs=epochs)

Epoch 1/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.6090 - loss: 0.7939
Epoch 2/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.7033 - loss: 0.5834
Epoch 3/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.7332 - loss: 0.5585
Epoch 4/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.7533 - loss: 0.5412
Epoch 5/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.7655 - loss: 0.5296
Epoch 6/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.7759 - loss: 0.5188
Epoch 7/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.7869 - loss: 0.5090
Epoch 8/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.7942 - loss: 0.4962
Epoch 9/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.7988 - loss: 0.4890
Epoch 10/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.8057 - loss: 0.4811
Epoch 11/1000
2069/2069 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.8094 - loss: 0.4754
E

In [14]:
score = model.evaluate(X_test, t_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.467856764793396
Test accuracy: 0.8361778259277344
